# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0369/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Loading Data
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")



README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
# Connecting DuckDB
import duckdb
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
print("DuckDB connection ready.")

DuckDB connection ready.


## 1. Method choice and why

I will use Logistic Regression as the first modeling method.

My Week-4 baseline is a hand-written rule based mainly on search position, CTR, and impressions. Logistic Regression is a suitable next step because it can learn how combinations of these signals relate to the action label without introducing unnecessary model complexity.

I chose it instead of starting with a more complex model because the goal is to establish a clear and interpretable model baseline first. The model should improve decision support rather than simply become more complicated.

The model will use only information available at the prediction point. I will not use trend_pct, trend_direction, is_declining_label, or other future/label-derived fields as features.

## 2. Split design

I will use a time-based split so that earlier observations are used for training and later observations are held out for evaluation.

The split will be based on report_date. This is preferable to a random split because the task is intended to represent a future decision: learn from earlier observations and evaluate on later observations.

The same evaluation window and metric will be used for both the ML model and the Week-4 baseline.

I will also avoid using any future-window or label-derived fields.

In [4]:
# March Data
march_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {REL}
""").df()

march_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [5]:
ml_data = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position,

    MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
        AS gsc_available

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("ML rows:", len(ml_data))
print("Columns:", ml_data.columns.tolist())

ml_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML rows: 176738
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'gsc_available']


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,gsc_available
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.175439,4.394234,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,7.842593,1
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,8.454069,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.422238,6.320337,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.577617,4.459107,1


In [6]:
duplicate_check = (
    ml_data
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
)

print("Rows:", len(ml_data))
print("Duplicate client-content combinations:",
      (duplicate_check > 1).sum())

Rows: 176738
Duplicate client-content combinations: 0


In [7]:
ml_data[[
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]].isna().sum()

,0
impressions,0
clicks,0
ctr,0
avg_position,1434


In [8]:
ml_data[[
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]].describe()

,impressions,clicks,ctr,avg_position
count,176738.000000,176738.000000,176738.000000,175304.000000
mean,1587.986675,4.650002,0.459397,17.050555
std,5431.337724,26.722649,3.775992,18.333942
min,1.000000,0.000000,0.000000,0.101639
25%,20.000000,0.000000,0.000000,5.500000
50%,173.000000,0.000000,0.000000,9.000000
75%,1039.000000,2.000000,0.215796,22.000000
max,617124.000000,5668.000000,100.000000,309.000000


In [9]:
feature_cols = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

X = ml_data[feature_cols].copy()

print("Features:")
print(feature_cols)

X.head()

Features:
['impressions', 'clicks', 'ctr', 'avg_position']


,impressions,clicks,ctr,avg_position
0,1140.0,2.0,0.175439,4.394234
1,57.0,0.0,0.000000,7.842593
2,149.0,0.0,0.000000,8.454069
3,1421.0,6.0,0.422238,6.320337
4,2770.0,16.0,0.577617,4.459107


In [10]:
# April data
APR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

print("April source configured.")

April source configured.


In [11]:
# March daily features
march_daily = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position

FROM {REL}

WHERE gsc_data_available IS TRUE

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
""").df()

print("March daily rows:", len(march_daily))
march_daily.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March daily rows: 3611061


,report_date,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,2026-03-01,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,4.0,0.0,0.0,3.50000
1,2026-03-01,client_62f4a7e64f5e0096,content_ac8663da7484669a,8.0,0.0,0.0,5.87500
2,2026-03-01,client_62f4a7e64f5e0096,content_d49a012dcb924e31,5.0,0.0,0.0,1.40000
3,2026-03-01,client_62f4a7e64f5e0096,content_614baf2af4330bd7,21.0,0.0,0.0,3.47619
4,2026-03-01,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,1.0,0.0,0.0,6.00000


In [12]:
# April 1–7 outcome data
april_daily = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
            ELSE NULL
        END
    ) AS avg_position

FROM {APR_REL}

WHERE
    gsc_data_available IS TRUE
    AND report_date <= DATE '2026-04-07'

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
""").df()

print("April outcome rows:", len(april_daily))
april_daily.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April outcome rows: 880921


,report_date,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,2026-04-01,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,1.0,0.0,0.0,NaN
1,2026-04-01,client_62f4a7e64f5e0096,content_f1c085e5ea530266,5.0,0.0,0.0,19.800000
2,2026-04-01,client_62f4a7e64f5e0096,content_c4901b51a12b1c3b,6.0,0.0,0.0,4.833333
3,2026-04-01,client_62f4a7e64f5e0096,content_e68b30ce53db783f,5.0,0.0,0.0,1.800000
4,2026-04-01,client_62f4a7e64f5e0096,content_dd37ba6019bb33f7,1.0,0.0,0.0,3.000000


In [14]:
#Combine March & April
import pandas as pd
all_daily = pd.concat(
    [march_daily, april_daily],
    ignore_index=True
)

all_daily = all_daily.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

print("Combined rows:", len(all_daily))
print("First date:", all_daily["report_date"].min())
print("Last date:", all_daily["report_date"].max())

Combined rows: 4491982
First date: 2026-03-01 00:00:00
Last date: 2026-04-07 00:00:00


In [20]:
# Create the future 7-day label
con.register("daily_data", all_daily)

future_labels = con.sql("""
    SELECT
        a.report_date,
        a.client_hash_id,
        a.content_hash_id,
        CASE
            WHEN MAX(
                CASE
                    WHEN b.avg_position <= 10
                         AND b.ctr < 2
                    THEN 1
                    ELSE 0
                END
            ) = 1
            THEN 1
            ELSE 0
        END AS future_low_ctr_strong_position
    FROM daily_data a
    LEFT JOIN daily_data b
        ON a.client_hash_id = b.client_hash_id
        AND a.content_hash_id = b.content_hash_id
        AND b.report_date > a.report_date
        AND b.report_date <= a.report_date + INTERVAL 7 DAY
    WHERE a.report_date >= DATE '2026-03-01'
      AND a.report_date < DATE '2026-04-01'
    GROUP BY
        a.report_date,
        a.client_hash_id,
        a.content_hash_id
""").df()

future_labels.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,future_low_ctr_strong_position
0,2026-03-26,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0
1,2026-03-31,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0
2,2026-03-29,client_0797ff3a1fc9a6a5,content_12890868e4cdac06,1
3,2026-03-31,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,1
4,2026-03-31,client_0797ff3a1fc9a6a5,content_20346a450ede60c6,1


In [21]:
all_daily = all_daily.drop(
    columns=["future_low_ctr_strong_position"],
    errors="ignore"
)

all_daily = all_daily.merge(
    future_labels,
    on=["report_date", "client_hash_id", "content_hash_id"],
    how="left"
)

all_daily["future_low_ctr_strong_position"] = (
    all_daily["future_low_ctr_strong_position"]
    .fillna(0)
    .astype(int)
)

print(
    all_daily["future_low_ctr_strong_position"]
    .value_counts()
)

future_low_ctr_strong_position
1    2724270
0    1767712
Name: count, dtype: int64


In [22]:
model_data = all_daily[
    (all_daily["report_date"] >= "2026-03-01") &
    (all_daily["report_date"] <= "2026-03-31")
].copy()

model_data = model_data[
    model_data["report_date"] <= "2026-03-31"
].copy()

print("Model rows:", len(model_data))
print(
    model_data["future_low_ctr_strong_position"]
    .value_counts()
)

Model rows: 3611061
future_low_ctr_strong_position
1    2724270
0     886791
Name: count, dtype: int64


In [23]:
print(
    model_data[
        "future_low_ctr_strong_position"
    ].value_counts(dropna=False)
)

future_low_ctr_strong_position
1    2724270
0     886791
Name: count, dtype: int64


In [24]:
model_data.groupby("report_date")["future_low_ctr_strong_position"].agg(
    ["count", "sum", "mean"]
).head(10)

,count,sum,mean
report_date,,,
2026-03-01,101910,79854,0.783574
2026-03-02,103696,79731,0.768892
2026-03-03,107362,80617,0.750890
2026-03-04,109377,81027,0.740805
2026-03-05,109740,81267,0.740541
2026-03-06,110037,81879,0.744104
2026-03-07,102153,77821,0.761808
2026-03-08,101170,77446,0.765504
2026-03-09,111313,84534,0.759426


In [25]:
model_data.groupby("report_date")["future_low_ctr_strong_position"].agg(
    ["count", "sum", "mean"]
).tail(10)

,count,sum,mean
report_date,,,
2026-03-22,122193,91528,0.749045
2026-03-23,122515,92105,0.751785
2026-03-24,124920,93626,0.749488
2026-03-25,126067,95061,0.754051
2026-03-26,126659,95755,0.756006
2026-03-27,125528,95489,0.760699
2026-03-28,122821,93683,0.762760
2026-03-29,123214,93452,0.758453
2026-03-30,125504,94204,0.750606


In [26]:
print(model_data["report_date"].min())
print(model_data["report_date"].max())

2026-03-01 00:00:00
2026-03-31 00:00:00


## 3. Train + compare vs my baseline

First create the features.

In [27]:
feature_cols = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position"
]

model_data = model_data.dropna(
    subset=["avg_position"]
).copy()

X = model_data[feature_cols].copy()

y = model_data[
    "future_low_ctr_strong_position"
].astype(int)

print("Features:", feature_cols)
print("Rows:", len(model_data))
print("Positive outcomes:", y.sum())
print("Negative outcomes:", (y == 0).sum())


Features: ['impressions', 'clicks', 'ctr', 'avg_position']
Rows: 3447872
Positive outcomes: 2594948
Negative outcomes: 852924


In [28]:
#Time split
train_mask = model_data["report_date"] < "2026-03-22"
test_mask = model_data["report_date"] >= "2026-03-22"

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positives:", y_train.sum())
print("Test positives:", y_test.sum())

Training rows: 2262711
Test rows: 1185161
Training positives: 1703586
Test positives: 891362


In [29]:
# Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [30]:
#Predictions
model_data.loc[test_mask, "model_probability"] = (
    model.predict_proba(X_test)[:, 1]
)

model_data.loc[test_mask, "model_prediction"] = (
    model.predict(X_test)
)

model_data.loc[test_mask, "model_prediction"] = (
    model_data.loc[test_mask, "model_prediction"].astype(int)
)

model_data[test_mask][[
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "ctr",
    "avg_position",
    "future_low_ctr_strong_position",
    "model_probability",
    "model_prediction"
]].head(20)

,report_date,client_hash_id,content_hash_id,ctr,avg_position,future_low_ctr_strong_position,model_probability,model_prediction
0,2026-03-26,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0.000000,9.000000,0,0.720512,1.0
25,2026-03-22,client_0797ff3a1fc9a6a5,content_04c67f3541177192,11.111111,15.666667,0,0.569384,1.0
26,2026-03-23,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,19.285714,0,0.495307,0.0
27,2026-03-24,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,21.000000,0,0.455720,0.0
28,2026-03-25,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,17.133333,0,0.545077,1.0
29,2026-03-26,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,17.565217,0,0.533745,1.0
30,2026-03-27,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,13.071429,0,0.636310,1.0
31,2026-03-28,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,16.645161,0,0.553631,1.0
32,2026-03-29,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,15.000000,0,0.594354,1.0
33,2026-03-30,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0.000000,21.250000,0,0.450797,0.0


In [35]:
# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_true = y_test

y_pred = model_data.loc[
    test_mask,
    "model_prediction"
].astype(int)

y_prob = model_data.loc[
    test_mask,
    "model_probability"
].astype(float)

model_accuracy = accuracy_score(y_true, y_pred)
model_precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)
model_recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)
model_f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)
model_auc = roc_auc_score(
    y_true,
    y_prob
)

print("Model evaluation")
print("----------------")
print(f"Accuracy : {model_accuracy:.4f}")
print(f"Precision: {model_precision:.4f}")
print(f"Recall   : {model_recall:.4f}")
print(f"F1       : {model_f1:.4f}")
print(f"ROC-AUC  : {model_auc:.4f}")

Model evaluation
----------------
Accuracy : 0.8307
Precision: 0.9082
Recall   : 0.8620
F1       : 0.8845
ROC-AUC  : 0.8612


In [38]:
# Getting old CSV
baseline = pd.read_csv(
    "/content/baseline_action_score.csv"
)

print("Baseline rows:", len(baseline))
print(baseline.columns.tolist())
baseline.head()

Baseline rows: 91723
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'position_score', 'ctr_score', 'volume_score', 'score', 'reason_code', 'action']


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_score,ctr_score,volume_score,score,reason_code,action
0,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142,2.56,40,40,10,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
1,client_e547b89c05043229,content_545bb6cc7081ded3,122905.0,287.0,0.234,2.62,40,40,10,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
2,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,0.103,2.48,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
3,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,0.043,1.49,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
4,client_73cda7b4e4f265ea,content_80eb6221de550658,79766.0,175.0,0.219,2.23,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW


In [39]:
# Checking old CSV
baseline_test = model_data.loc[test_mask].copy()

baseline_test["baseline_prediction"] = (
    (baseline_test["avg_position"] <= 10) &
    (baseline_test["ctr"] < 2)
).astype(int)

print("Baseline test rows:", len(baseline_test))
print("Baseline positive predictions:",
      baseline_test["baseline_prediction"].sum())
print("Actual positive outcomes:",
      baseline_test["future_low_ctr_strong_position"].sum())

Baseline test rows: 1185161
Baseline positive predictions: 660262
Actual positive outcomes: 891362


In [40]:
# Evaluating Baseline Score CSV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

baseline_true = baseline_test[
    "future_low_ctr_strong_position"
].astype(int)

baseline_pred = baseline_test[
    "baseline_prediction"
].astype(int)

baseline_accuracy = accuracy_score(
    baseline_true,
    baseline_pred
)

baseline_precision = precision_score(
    baseline_true,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    baseline_true,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    baseline_true,
    baseline_pred,
    zero_division=0
)

print("Week-4 baseline evaluation")
print("--------------------------")
print(f"Accuracy : {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"F1       : {baseline_f1:.4f}")

Week-4 baseline evaluation
--------------------------
Accuracy : 0.7481
Precision: 0.9489
Recall   : 0.7029
F1       : 0.8076


### Model versus baseline interpretation

The Logistic Regression model performed better than the Week-4 baseline on accuracy, recall, and F1 score.

| Metric | Week-4 Baseline | Logistic Regression |
|---|---:|---:|
| Accuracy | 0.7481 | 0.8307 |
| Precision | 0.9489 | 0.9082 |
| Recall | 0.7029 | 0.8620 |
| F1 | 0.8076 | 0.8845 |

The largest improvement was in recall, which increased from 0.7029 to 0.8620. This means the model identified more of the future positive cases than the original rule. F1 also increased from 0.8076 to 0.8845, showing a better balance between precision and recall.

The baseline had higher precision, 0.9489 compared with 0.9082 for the Logistic Regression model. Therefore, the baseline produced fewer false positives among the cases it flagged, while the model accepted some additional false positives in exchange for finding more true positive cases.

The model's 984,492 correct predictions, 123,017 false negatives, and 77,652 false positives also show that errors remain. The model should therefore be treated as decision support rather than an automatic decision maker.

Overall, the Logistic Regression model is a useful improvement over the Week-4 baseline for this task because it achieves higher recall and F1 on the same March 22–31 test period. The improvement comes with a small reduction in precision, which is an important trade-off to keep in mind.

In [41]:
# Comparing both
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],
    "Week-4 Baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],
    "Logistic Regression": [
        model_accuracy,
        model_precision,
        model_recall,
        model_f1
    ]
})

comparison

,Metric,Week-4 Baseline,Logistic Regression
0,Accuracy,0.748065,0.830682
1,Precision,0.948896,0.908212
2,Recall,0.702879,0.861990
3,F1,0.807567,0.884498


## 4. Errors and interpretation

The model was tested on the March 22–31 period using the same future outcome as the Week-4 baseline.

The model made 984,492 correct predictions, 123,017 false negative predictions, and 77,652 false positive predictions.

A false positive means the model predicted that the future low-CTR/strong-position condition would happen, but the condition did not occur in the following seven days. A false negative means the model missed a condition that did occur in the following seven days.

There were more false negatives than false positives. This means the model still missed a noticeable number of future cases, so it should not be treated as a final decision maker.

The feature coefficients show that average position had the strongest relationship with the model prediction, with a coefficient of -1.837. Impressions had the next strongest coefficient at -0.161, followed by clicks at 0.103 and CTR at -0.032.

Because the features were standardized before training, the coefficient sizes help show their relative influence in this model. The negative coefficient for average position means that higher average-position values were associated with a lower predicted probability of the future flag in this model. This does not mean that average position causes the outcome.

The errors also show a limitation of using only four current performance signals. Some future outcomes were missed, while some predicted problems did not happen. The model is therefore best treated as decision support and as a step beyond the Week-4 rule, rather than as a perfect predictor.

In [42]:
# Calculating False Positives and False Negatives
test_results = model_data.loc[test_mask].copy()

test_results["actual"] = (
    test_results["future_low_ctr_strong_position"]
    .astype(int)
)

test_results["predicted"] = (
    test_results["model_prediction"]
    .astype(int)
)

test_results["error_type"] = "Correct"

test_results.loc[
    (test_results["actual"] == 1) &
    (test_results["predicted"] == 0),
    "error_type"
] = "False Negative"

test_results.loc[
    (test_results["actual"] == 0) &
    (test_results["predicted"] == 1),
    "error_type"
] = "False Positive"

print(
    test_results["error_type"]
    .value_counts()
)


error_type
Correct           984492
False Negative    123017
False Positive     77652
Name: count, dtype: int64


In [43]:
# False Positives
false_positives = test_results[
    test_results["error_type"] == "False Positive"
].copy()

false_positives[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "avg_position",
        "future_low_ctr_strong_position",
        "model_probability",
        "model_prediction"
    ]
].sort_values(
    "model_probability",
    ascending=False
).head(10)

,report_date,client_hash_id,content_hash_id,ctr,avg_position,future_low_ctr_strong_position,model_probability,model_prediction
765451,2026-03-31,client_20259bd6705d81d4,content_fa4b9e9229816684,4.033546,6.558706,0,0.999999,1.0
1293442,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,1.832175,25.035826,0,0.999989,1.0
681827,2026-03-23,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,7.393365,4.180095,0,0.999059,1.0
471967,2026-03-23,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,4.674556,1.928402,0,0.998949,1.0
1074565,2026-03-31,client_23a62021009f63c4,content_74de5f247659e956,1.558632,18.572293,0,0.998524,1.0
681828,2026-03-24,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,5.938494,3.772004,0,0.995037,1.0
471966,2026-03-22,client_0fa64a184f18a4a0,content_9a4459a8a3b7a514,3.611111,1.732639,0,0.992250,1.0
681830,2026-03-26,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,5.896806,4.068796,0,0.991057,1.0
681829,2026-03-25,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,5.399061,3.604460,0,0.989723,1.0
681834,2026-03-30,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,6.424581,5.139665,0,0.989074,1.0


In [44]:
# False Negatives
false_negatives = test_results[
    test_results["error_type"] == "False Negative"
].copy()

false_negatives[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "avg_position",
        "future_low_ctr_strong_position",
        "model_probability",
        "model_prediction"
    ]
].sort_values(
    "model_probability",
    ascending=True
).head(10)

,report_date,client_hash_id,content_hash_id,ctr,avg_position,future_low_ctr_strong_position,model_probability,model_prediction
892904,2026-03-22,client_23a62021009f63c4,content_13ef8874a9a1ef5e,0.000000,497.00000,1,4.936530e-20,0.0
1074600,2026-03-31,client_23a62021009f63c4,content_74e1dddeda79c23c,0.000000,444.00000,1,6.837705e-18,0.0
4127433,2026-03-31,client_fef1a8f436438636,content_1100deb13f2922e0,0.000000,363.00000,1,1.281597e-14,0.0
1325039,2026-03-30,client_23a62021009f63c4,content_f7b549e701398ffc,0.000000,328.00000,1,3.326151e-13,0.0
972952,2026-03-31,client_23a62021009f63c4,content_3f82b120d085d740,0.000000,296.00000,1,6.525613e-12,0.0
984506,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,0.002495,0.08335,1,9.728941e-12,0.0
1540871,2026-03-22,client_3ffa76342f366962,content_b399885aea5f7e14,0.000000,291.00000,1,1.039785e-11,0.0
434055,2026-03-25,client_08a6a72ff48e62c0,content_f595aa4d56698626,0.000000,289.00000,1,1.252432e-11,0.0
663831,2026-03-25,client_20259bd6705d81d4,content_4fbbd7b748213ab6,0.000000,289.00000,1,1.252432e-11,0.0
1517149,2026-03-22,client_3ffa76342f366962,content_344ee596894372a1,0.000000,286.00000,1,1.655658e-11,0.0


### Model feature interpretation
The coefficients give a directional indication of which standardized input signals the model relied on most. They should not be interpreted as causal effects.

I will use this interpretation to check whether the model is relying on signals that make practical sense for the search-performance problem.

In [45]:

classifier = model.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": classifier.coef_[0]
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

,feature,coefficient,absolute_coefficient
3,avg_position,-1.837370,1.837370
0,impressions,-0.160529,0.160529
1,clicks,0.102833,0.102833
2,ctr,-0.032021,0.032021


Because I used StandardScaler, the coefficients can be compared by their absolute values. The larger the absolute coefficient, the more influence that feature had on the model.

Average position was by far the strongest feature in the Logistic Regression model. Its absolute coefficient of 1.84 was much larger than the other features. Impressions and clicks had relatively small effects, while CTR had almost no additional influence in this model. The negative coefficient for average position means that, as the position value increases, the probability of the positive target generally decreases.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.